# Corpus FR — générer les dialogues manquants (profonds, sociaux, EN, code-switch) → Hub

**Runtime : CPU suffit.** Secrets : `HF_TOKEN` (écriture), **`GEMINI_API_KEY`**.

L'audit du corpus l'a mesuré : **627 des 695 dialogues n'ont que deux tours** — presque
aucun historique à apprendre. Le générateur `lfm2-generate-fr` sait écrire ce qui manque
mais n'y a jamais été invité. Cette cellule génère quatre familles, chacune par **shards
de 300 poussés sur le Hub dès qu'ils existent** (`C_dialogues/v2_parts/`) — une VM
perdue ne coûte qu'un shard, jamais les appels Gemini déjà payés :

| famille | défaut | ce que ça corrige |
|---|---|---|
| `--deep` | 3000 | dialogues longs à reprises anaphoriques — l'historique |
| `--social` | 3000 | micro-échanges sociaux, le registre le plus faible de v3 |
| `--en` | 1200 | conversationnel anglais, la part de préservation (80/20) |
| `--switch` | 800 | code-switch, en plus des 196 existants |

Le fichier fusionné part sur `Rcarvalo/lfm25-fr-corpus-v1` en `C_dialogues/dialogues_v2.jsonl`.
Le verdict entre `===RESULT===` donne la **distribution des tours utilisateur** — c'est le
chiffre à faire bouger.

**Ensuite** : dans `colab_corpus_assistant_waves.ipynb`, `corpus/C_dialogues/dialogues_v2.jsonl`
est déjà dans `BRICK_A_SOURCES` — relancez-le pour faire parler ces dialogues par `fr_female`.

Coût : ~800 appels Gemini Flash, quelques euros au plus. Sur des 429, relancez la cellule :
les shards déjà sur le Hub sont réutilisés.


In [ ]:
# Jetons — le plus propre : Colab > icône clé > secrets HF_TOKEN (écriture), GEMINI_API_KEY, WANDB_API_KEY (optionnel).
import os
from getpass import getpass
try:
    from google.colab import userdata
    read = userdata.get
except Exception:
    read = lambda name: getpass(f"{name} : ")
for name, required in (("HF_TOKEN", True), ("GEMINI_API_KEY", False), ("WANDB_API_KEY", False)):
    try:
        value = read(name)
    except Exception:
        value = "" if not required else getpass(f"{name} : ")
    if value:
        os.environ[name] = value
    elif required:
        raise SystemExit(f"{name} manquant")
print("jetons chargés :", [n for n in ("HF_TOKEN", "GEMINI_API_KEY", "WANDB_API_KEY") if os.environ.get(n)])


## Générer et pousser (≈ 1-2 h CPU selon le quota Gemini) — relançable

In [ ]:
import os, subprocess, urllib.request
os.environ.update({
    "LFM2_BRANCH": "rd/pr_rca_eval_baseline",
    "LFM2_JOB": "generate_dialogues_fr",
    "LFM2_ARGS": "--deep 3000 --social 3000 --en 1200 --switch 800",
    "LFM2_EXTRAS": "serving-liquid,eval,inspect",
    "LFM2_ROOT": "/content/repo",
    "LFM2_OUT": "/content/out"
})
os.makedirs("/content/out", exist_ok=True)
urllib.request.urlretrieve("https://raw.githubusercontent.com/rcarvalo/finetuning_s2s_toolcalling/rd/pr_rca_eval_baseline/infra/colab_entrypoint.sh", "/content/entry.sh")

# Au PREMIER PLAN, volontairement : le kernel occupé est ce qui garde la session Colab vivante.
# La sortie est streamée ligne à ligne ici ET dans /content/out/job.log ; les lignes ===RESULT===
# sont reprises en résumé à la fin, avec le VRAI code de sortie du job.
results = []
with subprocess.Popen(["bash", "/content/entry.sh"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
        if "===RESULT===" in line:
            results.append(line.strip())
code = proc.returncode
print("\n" + "=" * 70)
print("STATUT :", "SUCCÈS" if code == 0 else f"ÉCHEC (code {code}) — voir les dernières lignes ci-dessus")
for line in results:
    print("  ", line)
print("journal complet : /content/out/job.log")
